In [1]:
#0 pasa price forecast

#saved elsewhere

In [2]:
# 1. set up variables

#LOSSES = df_day['CLEAREDSUPPLY'] - df_day['TOTALDEMAND'] - df_day['DISPATCHABLELOAD']
#GENERATION = df_day['TOTALDEMAND'] + df_day['NETINTERCHANGE'] +df_day['DISPATCHABLELOAD']+df_day['LOSSES']


import pulp

import time
import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path
import os
from datetime import datetime
from zoneinfo import ZoneInfo



SQE_duid = ['BANGOWF1','BANGOWF2','CRURWF1','CLRKCWF1','CLRKCWF2','SAPHWF1','MUWAWF1','MUWAWF2']

IC_DATA = {
    'T_V_MNSP1':  {'name': 'Basslink',   'min': -400,  'max': 456,  'loss_max': 60, 'loss_factor': 0*0.04},
    'V_SA':       {'name': 'Heywood',    'min': -650,  'max': 600,  'loss_max': 100, 'loss_factor': 0.12},
    'V_S_MNSP1':  {'name': 'Murraylink', 'min': -200,  'max': 220,  'loss_max': 80, 'loss_factor': 0*0.15},
    'N_Q_MNSP1':  {'name': 'Terranora',  'min': -107,  'max': 210,  'loss_max': 90, 'loss_factor': 0*0.4},
    'NSW1_QLD1':  {'name': 'QNI',        'min': -1400, 'max': 850,  'loss_max': 250, 'loss_factor': 0.125},
    'VIC1_NSW1':  {'name': 'VIC_NSW',    'min': -1900, 'max': 1700, 'loss_max': 400, 'loss_factor': 0.20},
}

IC_list = list(IC_DATA.keys())

flow = {}
losses = {}
direction = {}

for ic, d in IC_DATA.items():

    flow[ic] = pulp.LpVariable(f"{d['name']}_flow",lowBound=d['min'],upBound=d['max'])
    losses[ic] = pulp.LpVariable(f"{d['name']}_losses",lowBound=0, upBound=d['loss_max'])
    direction[ic] = pulp.LpVariable(f"{d['name']}_direction", cat="Binary")



regions = ['NSW1','QLD1','SA1','TAS1','VIC1']
blk_coal_N, blk_coal_Q,brn_coal = 0,0,0

def losses_func(HH_demand):
    
    # calculate slopes y = Ax + B 
    A = [2*9.58*10**-5,2*1.15*10**-4,2*9.36*10**-4,2*2.075*10**-3,2*9.47*10**-5,2*8.59*10**-5]
    # these are duplicated above and should be tidied up
    B_BL = 0.0142 +(1.65*10**-6)*HH_demand[4]-(2.7447*10**-6 )*HH_demand[3]
    B_HW = -0.0258-HH_demand[4]*1.09*10**-5+HH_demand[2]*6.63*10**-5-HH_demand[0]*2.22*10**-6
    B_QNI = -0.06-HH_demand[0]*7.58*10**-7+HH_demand[1]*1.33*10**-5
    B_VN = 0.0763-HH_demand[4]*7.19*10**-6+HH_demand[0]*6.43*10**-6-HH_demand[2]*5.08*10**-5

    B = [B_BL,B_HW,-4.06*10**-2,7.9*10**-3,B_QNI,B_VN]

     #number of linear equations for each interconnector
    df_losseq_parts = []
    step = 20
    for IC in IC_list: 
        xinterval = (IC_DATA[IC]['max'] - IC_DATA[IC]['min'])/(step-1)
        IC_DATA[IC]['max']
        
        slope = []
        intercept = []
        
        for i in range(step):
            #increment by one interval
            x_value = IC_DATA[IC]['min'] + xinterval*i
            #differenticate equation for slope
            slope.append(A[IC_list.index(IC)]*x_value+B[IC_list.index(IC)])
            #substitue into full equation
            y_value = y_value_func(IC, x_value, HH_demand)
            #b= y-mx
            intercept.append(y_value - slope[i]*x_value) 
        df = pd.DataFrame(data ={'IC': IC, 'Slope': slope,'Intercept': intercept} )
        df_losseq_parts.append(df)
    df_losseq=pd.concat(df_losseq_parts, ignore_index=True)
    return df_losseq

def y_value_func(IC,flow,HH_demand): 
    if IC=='NSW1_QLD1':
        y_value  = (-0.1099+HH_demand[0]*3.8443*10**-6+HH_demand[1]*1.5932*10**-5)*flow + (flow**2)*8.9827*10**-5
    if IC=='N_Q_MNSP1':
        y_value =  -0.0013*flow + 2.7372*10**-4*(flow**2)
    if IC=='T_V_MNSP1':
        y_value = -flow*3.92*10**-3 + (1.0393*10**-4)*(flow**2) + 4
    if IC=='V_SA':
        y_value = (-0.0279-HH_demand[4]*1.4896*10**-5+HH_demand[2]*5.1080*10**-5+HH_demand[0]*1.6981*10**-6)*flow+(flow**2)*1.34*10**-4
       
    if IC=='V_S_MNSP1':
        y_value = 0.0039*flow + 2.8177*10**-4*(flow**2)
    if IC=='VIC1_NSW1':
        y_value = (0.0556-HH_demand[4]*6.6712*10**-6+HH_demand[0]*1.3820*10**-5-HH_demand[2]*6.0293*10**-5)*flow + (flow**2)*7.8803*10**-5
    return y_value



# this calculate the contraint outcomes
def get_constraints(t,p):

    rows = []
    
    for name, constraint in prob.constraints.items():
    
        rhs_value = -constraint.constant 
        lhs_value = constraint.value()  - constraint.constant         #constraint.value()  evaluates LHS - RHS
        dual_value = constraint.pi              # shadow price (marginal value)
    
        rows.append({
            "INTERVAL_DATETIME":t,
            "Constraint": name,
            "LHS": lhs_value,
            "RHS": rhs_value,
            "Marginal_price": dual_value
        })
    
    df_constraints = pd.DataFrame(rows)
    df_constraints = df_constraints[df_constraints.Marginal_price != 0]
    return df_constraints



def is_databricks():
    return "DATABRICKS_RUNTIME_VERSION" in os.environ



print("OK")



OK


In [3]:
#2.  build and solve the model

N_94T_947_72 = {"FLYCRKWF": -1.0,"NYNGAN1": 0.616, "NEVERSF1": 0.616, "WELLSF1": 0.616, "SUNTPSF1": 0.598,"MOLNGSF1": 0.554,"BODWF1":0.544,
                    "MANSLR1":0.531,"PARSF1":0.457,"GOONSF1":0.457,"QPSFB1":0.457,"QPSFB2":0.457,"WELNSF1":0.439,"ORABESS1":0.439,"BERYLSF1":0.41,
                   "JEMALNG1":0.387,"STUBSF1":0.271,"STUBSF2":0.271,"WOLARSF1":0.153}

N_NIL_94T =  {"MOLNGSF": 1.0,"MANSLR1": 0.873,"PARSF1": 0.475,"GOONSF1": 0.475,"QPSFB1": 0.475,"QPSFB2": 0.475 ,
                  "FLYCRKWF": -0.446,"JEMALNG1":0.402 ,"SUNTPSF":0.189 ,"NYNGAN1": 0.153,"NEVERSF1": 0.153,"WELLSF1": 0.153}

N_NIL_969 = {"GNNDHSF1": 1.0,"MOREESF1": 0.301}

N_NIL_060_051 = {'GESF1': +1,	'HUMENSW': +0.995,	'WLWLSF2': +0.995,	'WLWLSF1': +0.995,	'CUSF1': +0.992,	'CRWASF1': +0.976,\
                     'MULWASF1': +0.963,	'BLOWERNG': +0.947,	'WAGGNSF1': +0.947,	'JUNEESF1': +0.947,	'SEBSF1': +0.947,	'WSTWYSF1': +0.947,\
                     'WYASF1': +0.947,	'BOMENSF1': +0.947,	'FINLYSF1': +0.945,	'URANQ11': +0.944,	'URANQ12': +0.944,	'URANQ13': +0.944,\
                     'URANQ14': +0.944,	'AVLSF1': +0.917,	'COLEASF1': +0.883,	'HILLSTN1': +0.879,	'DARLSF1': +0.879,	'RIVNB2': +0.879,\
                     'RESS1': +0.879,	'DPNTB1': +0.879,	'LIMOSF21': +0.459,	'LIMOSF11': +0.459,	'SUNRSF1': +0.459,	'LIMBESS1': +0.459,\
                     'BROKENH1': +0.248,	'BHB1': +0.248,	'STWF1': +0.248,	'KARSF1': +0.207,	'YATSF1': +0.207,	'BANN1': +0.178,\
                     'WEMENSF1': +0.178,	'KIAMSF1': +0.172}

V_N_NIL_V2  =  {'DARTM1': -0.896,	'MCKAY1': -0.896,	'WKIEWA1': -0.896,	'WKIEWA2': -0.896,	'MURRAY': -0.786,	'ARWF1': -0.432,\
                    'BALB1': -0.432,	'YENDWF1': -0.432,	'BRYB1WF1': -0.432,	'BRYB2WF2': -0.432,	'BULGANA1': -0.432,	'BULBES1': -0.432,\
                    'RANGEB1': -0.432,	'CROWLWF1': -0.432,	'MERCER01': -0.432,	'MOORAWF1': -0.432,	'ELAINWF1': -0.432,	'GLENSF1': -0.432,\
                    'MOKOSF1': -0.432,	'GLRWNSF1': -0.432,	'WINTSF1': -0.432,	'MTGELWF1': -0.432,	'KIATAWF1': -0.432,	'HBESS1': -0.432,\
                    'GANNB1': -0.432,	'KERNGSP1': -0.432,	'GANNSF1': -0.432,	'COHUNSF1': -0.432,	'KIAMSF1': -0.432,	'KESSB1': -0.432,\
                    'VBB1': -0.432,	'MUWAWF1': -0.432,	'MUWAWF2': -0.432,	'PIBESS1': -0.432,	'BALDHWF1': -0.432,	'KARSF1': -0.432,\
                    'YATSF1': -0.432,	'NUMURSF1': -0.432,	'GIRGSF': -0.432,	'WUNUSF1': -0.432,	'LANCSF1': -0.432,	'CHYTWF1': -0.432,\
                    'MRNBESS1': -0.432,	'MRTLSWF1': -0.432,	'TRGBESS1': -0.432,	'SALTCRK1': -0.432,	'OAKLAND1': -0.432,	'HD1WF1': -0.432,\
                    'RYANCWF1': -0.432,	'MACARTH1': -0.432,	'BANN1': -0.432,	'WEMENSF1': -0.432,	'YWPS1': -0.36,	'YWPS2': -0.36,	'YWPS3': -0.36,\
                    'YWPS4': -0.36}

V_N_NIL_V1 = {'ALDGASF1': +1,	'BBATTERY1': +0.485,	'KIDSPHG1': +0.452,	'KIDSPHG2': +0.452,	'KIDSPHL1': -0.452,	'KIDSPHL2': -0.452,\
                  'BARRON-1': +0.452,	'BARRON-2': +0.452,	'DAYDSF1': +0.452,	'HAYMSF1': +0.452,	'YABULU2': +0.452,	'HAUGHT11': +0.452,\
                  'KAREEYA1': +0.452,	'KAREEYA2': +0.452,	'KAREEYA3': +0.452,	'KAREEYA4': +0.452,	'MSTUART1': +0.452,	'MSTUART2': +0.452,\
                  'MSTUART3': +0.452,	'KSP1': +0.452,	'RRSF1': +0.452,	'KEPWF1': +0.452,	'KEPSF1': +0.452,	'KABANWF1': +0.452,	\
                  'YABULU': +0.452,	'SMCSF1': +0.452,	'MEWF1': +0.452,	'CLARESF1': +0.451,	'CSPVPS1': +0.451,	'HAMISF1': +0.451,\
                  'WHITSF1': +0.451,	'BRDDSF01': +0.45,	'BRDDBES1': +0.45,	'CLRKCWF1': +0.45,	'CLRKCWF2': +0.45,	'RUGBYR1': +0.448,\
                  'Stanwell': +0.443,	'STAN-1': +0.443,	'STAN-2': +0.443,	'STAN-3': +0.443,	'STAN-4': +0.443,	'BARCALDN': +0.434,\
                  'LILYSF1': +0.434,	'CLERMSF1': +0.434,	'MIDDLSF1': +0.433,	'EMERASF1': +0.426,	'MOUSF1': +0.29,	'GSTONE1': -0.27,\
                  'GSTONE2': -0.27,	'GSTONE5': -0.27,	'GSTONE6': -0.27,	'CALL_B_1': +0.257,	'CALL_B_2': +0.257,	'CPP_3': +0.257,\
                  'CPP_4': +0.257,	'GSTONE3': -0.243,	'GSTONE4': -0.243,	'BUSF1': -0.172,	'CHILDSF1': -0.153}

Q_N_NIL_SRAR = {'SAPHWF1': -1.04}

V_NIL_MLGT_MLGT = {'VBB1': +1,	'MERCER01': +0.932,	'MOORAWF1': +0.932,	'ELAINWF1': +0.932,	'BRYB1WF1': +0.869,	'BRYB2WF2': +0.869,	\
                   'MRTLSWF1': +0.838,	'TRGBESS1': +0.838,	'ARWF1': +0.837,	'CROWLWF1': +0.829,	'BALB1': +0.824,	'YENDWF1': +0.822,\
                   'BULGANA1': +0.815,	'BULBES1': +0.815,	'KIATAWF1': +0.76,	'MTGELWF1': -0.735,	'MUWAWF1': +0.726,	'MUWAWF2': +0.726,\
                   'SALTCRK1': +0.685,	'OAKLAND1': +0.685,	'KIAMSF1': +0.598,	'GANNB1': +0.578,	'KERNGSP1': +0.578,	'GANNSF1': +0.578,\
                   'COHUNSF1': +0.578,	'KESSB1': +0.575,	'LNGS1': -0.564,	'LNGS2': -0.564,	'CRWARP1': +0.548,	'BANN1': +0.542,\
                   'WEMENSF1': +0.542,	'KARSF1': +0.529,	'YATSF1': +0.529,	'HD1WF1': +0.384,	'RYANCWF1': +0.384,	'MACARTH1': +0.384,\
                   'MLB01': +0.382,	'DUNDWF1': +0.382,	'DUNDWF2': +0.382,	'DUNDWF3': +0.382,	'MORTLK11': +0.382,	'MORTLK12': +0.382,\
                   'STOCKYD1': +0.381,	'GPWFWST1': +0.38,	'GPWFWST2': +0.38,	'GPWFEST1': +0.38,	'GPWFEST2': +0.38,	'GPWFEST3': +0.38,\
                   'BROKENH1': +0.358,	'BHB1': +0.358,	'STWF1': +0.358,	'NUMURSF1': +0.278,	'GIRGSF': +0.278,	'WUNUSF1': +0.278,\
                   'LANCSF1': +0.278,	'LIMOSF21': +0.255,	'LIMOSF11': +0.255,	'SUNRSF1': +0.255,	'LIMBESS1': +0.255,	'GOESF1': +0.24,\
                   'GLENSF1': +0.224,	'MOKOSF1': +0.22,	'GLRWNSF1': +0.22,	'WINTSF1': +0.22,	'NPS': -0.186,	'MREHA2': +0.155,\
                   'MREHA3': +0.155,	'MREHA1': +0.155,	'MURRAY': +0.153,	'HUMEV': +0.151}

V_NWVIC_GFT1_750 = {'ARWF1':1,'BULGANA1':1,'BULBES1':1,'CROWLWF1':1,'MUWAWF1':1,'MUWAWF2':1}

N_N_NIL_WGLT = {'HILLSTN1': +1,	'DARLSF1': +1,	'RIVNB2': +1,	'RESS1': +1,	'DPNTB1': +1,	'COLEASF1': +0.994,	'AVLSF1': +0.985,\
                'URANQ11': +0.972,	'URANQ12': +0.972,	'URANQ13': +0.972,	'URANQ14': +0.972,	'WAGGNSF1': +0.939,	'JUNEESF1': +0.939,\
                'SEBSF1': +0.939,	'WSTWYSF1': +0.939,	'WYASF1': +0.939,	'BOMENSF1': +0.939,	'FINLYSF1': +0.898,	'MULWASF1': +0.846,\
                'CUSF1': +0.812,	'CRWASF1': +0.81,	'WLWLSF2': +0.793,	'WLWLSF1': +0.793,	'HUMENSW': +0.756,	'GESF1': +0.736,\
                'LIMOSF21': +0.698,	'LIMOSF11': +0.698,	'SUNRSF1': +0.698,	'LIMBESS1': +0.698,	'BLOWERNG': +0.602,	'BROKENH1': +0.54,\
                'BHB1': +0.54,	'STWF1': +0.54,	'MURRAY': -0.291,	'GUNNING1': +0.216,	'BANGOWF1': +0.202,	'BANGOWF2': +0.202,	'KARSF1': +0.156,\
                'YATSF1': +0.156}

Q_NIL_LCCP_BCCP = {'ALDGASF1': +1,	'BBATTERY1': +0.485,	'KIDSPHG1': +0.452,	'KIDSPHG2': +0.452,	'KIDSPHL1': -0.452,	'KIDSPHL2': -0.452,\
                   'BARRON-1': +0.452,	'BARRON-2': +0.452,	'DAYDSF1': +0.452,	'HAYMSF1': +0.452,	'YABULU2': +0.452,	'HAUGHT11': +0.452,\
                   'KAREEYA1': +0.452,	'KAREEYA2': +0.452,	'KAREEYA3': +0.452,	'KAREEYA4': +0.452,	'MSTUART1': +0.452,	'MSTUART2': +0.452,\
                   'MSTUART3': +0.452,	'KSP1': +0.452,	'RRSF1': +0.452,	'KEPWF1': +0.452,	'KEPSF1': +0.452,	'KABANWF1': +0.452,\
                   'YABULU': +0.452,	'SMCSF1': +0.452,	'MEWF1': +0.452,	'CLARESF1': +0.451,	'CSPVPS1': +0.451,	'HAMISF1': +0.451,\
                   'WHITSF1': +0.451,	'BRDDSF01': +0.45,	'BRDDBES1': +0.45,	'CLRKCWF1': +0.45,	'CLRKCWF2': +0.45,	'RUGBYR1': +0.448,\
                   'Stanwell': +0.443,	'STAN-1': +0.443,	'STAN-2': +0.443,	'STAN-3': +0.443,	'STAN-4': +0.443,	'BARCALDN': +0.434,\
                   'LILYSF1': +0.434,	'CLERMSF1': +0.434,	'MIDDLSF1': +0.433,	'EMERASF1': +0.426,	'MOUSF1': +0.29,	'GSTONE1': -0.27,\
                   'GSTONE2': -0.27,	'GSTONE5': -0.27,	'GSTONE6': -0.27,	'CALL_B_1': +0.257,	'CALL_B_2': +0.257,	'CPP_3': +0.257,\
                   'CPP_4': +0.257,	'GSTONE3': -0.243,	'GSTONE4': -0.243,	'BUSF1': -0.172,	'CHILDSF1': -0.153}

Q_BCLC_BCCP_CLWU = {'BBATTERY1': +1,	'KIDSPHG1': +0.972,	'KIDSPHG2': +0.972,	'KIDSPHL1': -0.972,	'KIDSPHL2': -0.972,	'BARRON-1': +0.972,\
                    'BARRON-2': +0.972,	'DAYDSF1': +0.972,	'HAYMSF1': +0.972,	'YABULU2': +0.972,	'HAUGHT11': +0.972,	'KAREEYA1': +0.972,\
                    'KAREEYA2': +0.972,	'KAREEYA3': +0.972,	'KAREEYA4': +0.972,	'MSTUART1': +0.972,	'MSTUART2': +0.972,	'MSTUART3': +0.972,\
                    'KSP1': +0.972,	'RRSF1': +0.972,	'KEPWF1': +0.972,	'KEPSF1': +0.972,	'KABANWF1': +0.972,	'YABULU': +0.972,\
                    'SMCSF1': +0.972,	'MEWF1': +0.972,	'CLARESF1': +0.971,	'CSPVPS1': +0.971,	'HAMISF1': +0.971,	'WHITSF1': +0.971,\
                    'BRDDSF01': +0.97,	'BRDDBES1': +0.97,	'CLRKCWF1': +0.97,	'CLRKCWF2': +0.97,	'RUGBYR1': +0.968,	'STABESS1': +0.964,\
                    'STAN-1': +0.964,	'STAN-2': +0.964,	'STAN-3': +0.964,	'STAN-4': +0.964,	'LILYSF1': +0.957,	'BARCALDN': +0.956,\
                    'MIDDLSF1': +0.956,	'CLERMSF1': +0.956,	'EMERASF1': +0.95,	'MOUSF1': +0.833,	'CALL_B_1': +0.804,	'CALL_B_2': +0.804,\
                    'CPP_3': +0.804,	'CPP_4': +0.804,	'GSTONE1': -0.789,	'GSTONE2': -0.789,	'GSTONE5': -0.789,	'GSTONE6': -0.789,\
                    'ALDGASF1': -0.789,	'GSTONE3': -0.779,	'GSTONE4': -0.779,	'BUSF1': -0.525,	'CHILDSF1': -0.478,	'SRSF1': -0.457,\
                    'WOOLES1': -0.319,	'MUCRKSF1': -0.319,	'WOOLGSF1': -0.319}

def prob_solve(DI_currrent,s,load):
    #set up LP problem
    prob = pulp.LpProblem("Bid_Stack_Optimisation", pulp.LpMinimize)

    v = {bid_id: pulp.LpVariable(f"ENERGY_{bid_id}", lowBound=0, upBound=float(s.loc[bid_id, "VOL"]))     # --- LP variables: dispatch per bid (0..VOLUME)
        for bid_id in s.index}

    l = {bid_id: pulp.LpVariable(f"LOAD_{bid_id}", lowBound=float(load.loc[bid_id, "VOL"]), upBound=0)
        for bid_id in load.index}

    #Objective function
    obj = (pulp.lpSum(v[b] * float(s.loc[b, "PRICE"]) for b in s.index) + pulp.lpSum(l[a] * float(load.loc[a, "PRICE"]) for a in load.index)) \
    
    prob += obj
    
    prob.objective.name = "Total_dispatch_cost"   # optional, depending on PuLP version

    df_temp = df_pasa[df_pasa.SETTLEMENTDATE == DI_current]
    df_temp = df_temp[['REGIONID','TOTALDEMAND']]

    demand_by_region = (df_temp.set_index("REGIONID")["TOTALDEMAND"].to_dict())

    # ─── BUILD LOOKUP INDICES (once, at the top of prob_solve after s and load are final) ───

    from collections import defaultdict
    
    # DUID → list of bid indices (for network constraints)
    duid_to_indices = defaultdict(list)
    for b in s.index:
        duid_to_indices[s.loc[b, "DUID"]].append(b)
    
    # REGIONID → gen indices (for energy balance & ramping)
    region_gen = defaultdict(list)
    for b in s.index:
        region_gen[s.loc[b, "REGIONID"]].append(b)
    
    # REGIONID → load indices
    region_load = defaultdict(list)
    for b in load.index:
        region_load[load.loc[b, "REGIONID"]].append(b)
    
    # (FUEL, REGIONID) → gen indices (for ramping)
    fuel_region_gen = defaultdict(list)
    for b in s.index:
        fuel_region_gen[(s.loc[b, "FUEL"], s.loc[b, "REGIONID"])].append(b)
    
    # FUEL → gen indices (for brown coal ramping)
    fuel_gen = defaultdict(list)
    for b in s.index:
        fuel_gen[s.loc[b, "FUEL"]].append(b)
    
    
    # ─── ENERGY BALANCE CONSTRAINTS ───
    
    for region in regions:
        lhs = (pulp.lpSum(v[c] for c in region_gen[region]) + pulp.lpSum(l[d] for d in region_load[region]))
         
        if region == "NSW1":
            rhs = (flow['N_Q_MNSP1'] + flow['NSW1_QLD1'] - flow['VIC1_NSW1']) + 0.4394*losses['NSW1_QLD1']+0.5028*losses['N_Q_MNSP1']+(1-0.6262)*losses['VIC1_NSW1'] 
        elif region == "QLD1":
            rhs = (- flow['N_Q_MNSP1'] - flow['NSW1_QLD1']) +(1-0.4394)*losses['NSW1_QLD1']+(1-0.5028)*losses['N_Q_MNSP1']
        elif region == "SA1":
            rhs = (- flow['V_S_MNSP1'] - flow['V_SA']) + (1-0.4925)*losses['V_SA']+(1-0.5057)*losses['V_S_MNSP1']
        elif region == "TAS1":
            rhs = (flow['T_V_MNSP1'])  +  losses['T_V_MNSP1']
        else:
            rhs = (flow['VIC1_NSW1'] + flow['V_SA'] + flow['V_S_MNSP1'] - flow['T_V_MNSP1']) + 0.4925*losses['V_SA']+0.6262*losses['VIC1_NSW1']+0.5057*losses['V_S_MNSP1']
    
        prob += (lhs == demand_by_region[region] + rhs, f"{region}_energy_balance")
    
    
    # ─── RAMPING CONSTRAINTS ───
    
    if blk_coal_Q > 0:
        prob += pulp.lpSum(v[b] for b in fuel_region_gen[("Black_Coal", "QLD1")]) <= 1.18 * blk_coal_Q, "Black_coal_ramping_up_Q"
        prob += pulp.lpSum(v[b] for b in fuel_region_gen[("Black_Coal", "QLD1")]) >= 0.82 * blk_coal_Q, "Black_coal_ramping_down_Q"
    
    if blk_coal_N > 0:
        prob += pulp.lpSum(v[b] for b in fuel_region_gen[("Black_Coal", "NSW1")]) <= 1.18 * blk_coal_N, "Black_coal_ramping_up_N"
        prob += pulp.lpSum(v[b] for b in fuel_region_gen[("Black_Coal", "NSW1")]) >= 0.82 * blk_coal_N, "Black_coal_ramping_down_N"
    
    if brn_coal > 0:
        prob += pulp.lpSum(v[b] for b in fuel_gen["Brown_Coal"]) <= 1.18 * brn_coal, "Brown_coal_ramping_up"
        prob += pulp.lpSum(v[b] for b in fuel_gen["Brown_Coal"]) >= 0.82 * brn_coal, "Brown_coal_ramping_down"
    
    
    # ─── NETWORK CONSTRAINTS (all use the same pattern) ───
    #top 4 binding thermal constraints  in April 2026
   
    prob += pulp.lpSum(coeff * v[b] for duid, coeff in N_94T_947_72.items() for b in duid_to_indices.get(duid, [])) <= 475, "N>94T_947_72"

    prob += pulp.lpSum(coeff * v[b] for duid, coeff in N_NIL_94T.items() for b in duid_to_indices.get(duid, [])) <= 180, "N>NIL_94T"

    prob += pulp.lpSum(coeff * v[b] for duid, coeff in N_NIL_969.items() for b in duid_to_indices.get(duid, [])
    ) <= 110, "N>NIL_969"

    prob += pulp.lpSum(coeff * v[b] for duid, coeff in N_NIL_060_051.items() for b in duid_to_indices.get(duid, [])
    ) - 0.207 * flow['V_S_MNSP1'] <= 1300, "N>>NIL_060_051"
    #consider Q_STR_7C0K_MEWF_16,  N>9GL_999_998,N>>BDBU_060_051, S>NIL_HUWT_STBG
    #VIC-NSW limits
    #transient stability
    
    prob += pulp.lpSum(coeff * v[b] for duid, coeff in V_N_NIL_V2.items() for b in duid_to_indices.get(duid, [])
    ) + 1 * flow['VIC1_NSW1'] + 0.492 * flow['V_S_MNSP1'] + 0.478 * flow['V_SA'] - 0.398 * flow['T_V_MNSP1'] <= 0, "V::N_NIL_V2"
    #voltage
    
    prob += pulp.lpSum(coeff * v[b] for duid, coeff in V_N_NIL_V1.items() for b in duid_to_indices.get(duid, [])
    ) + 1 * flow['VIC1_NSW1'] + 0.192 * flow['V_S_MNSP1'] <= 1150, "V^^N_NIL_V1"

    #NSW_QLD limits & Sapphire
    
    prob += pulp.lpSum(coeff * v[b] for duid, coeff in Q_N_NIL_SRAR.items() for b in duid_to_indices.get(duid, [])
    ) - 1 * flow['NSW1_QLD1'] <= 1200, "Q^^N_NIL_SRAR"
    #Murra Warra 1 and 2
    
    prob += pulp.lpSum(coeff * v[b] for duid, coeff in V_NIL_MLGT_MLGT.items() for b in duid_to_indices.get(duid, [])
    ) - 0.159 * flow['VIC1_NSW1'] - 0.529 * flow['V_S_MNSP1'] - 0.433 * flow['V_SA'] <= 2200, "V>>NIL_MLGT_MLGT"

    
    prob += pulp.lpSum(coeff * v[b] for duid, coeff in V_NWVIC_GFT1_750.items() for b in duid_to_indices.get(duid, [])
    ) <= 500, "V_NWVIC_GFT1_750"
    #Bango
    
    prob += pulp.lpSum(coeff * v[b] for duid, coeff in N_N_NIL_WGLT.items() for b in duid_to_indices.get(duid, [])
    ) + 0.376 * flow['VIC1_NSW1'] - 0.156 * flow['V_S_MNSP1'] <= 1050, "N^^N_NIL_WGLT"
    #Clarke Creek
    
    prob += pulp.lpSum(coeff * v[b] for duid, coeff in Q_NIL_LCCP_BCCP.items() for b in duid_to_indices.get(duid, [])
    ) <= 1250, "Q>NIL_LCCP_BCCP"

    prob += pulp.lpSum(coeff * v[b] for duid, coeff in Q_BCLC_BCCP_CLWU.items() for b in duid_to_indices.get(duid, [])
    ) <= 1500, "Q>BCLC_BCCP_CLWU"
        
    #Interconnector losses
    df_losseq = losses_func(list(demand_by_region.values()))
    counter = defaultdict(int)
    
    constraints = df_losseq.to_numpy()
    
    for IC, slope, intercept in constraints:
     
        counter[IC] += 1
        n = counter[IC]
    
        prob += (losses[IC] >= slope * flow[IC] + intercept, f"{IC}_losses_{n}")
    
    
    #insert IC loss burn constraints
   

    h = DI_current.hour

    if h in [10,11, 12, 13, 14,15,16]:
        factor = 0.5
    else:
        factor = None
    
    if factor is not None:
        for ic, d in IC_DATA.items():
            prob += (losses[ic] <= d['loss_max'] * factor,f"Burn_{ic}_losses_H{h}" )
        
    
    # Solve
    status = prob.solve()
    print(str(DI_current),pulp.LpStatus[status],end = ' ')
   
    interchange = []

    interchange = [-flow['NSW1_QLD1'].varValue -flow['N_Q_MNSP1'].varValue +flow['VIC1_NSW1'].varValue,\
    flow['NSW1_QLD1'].varValue +flow['N_Q_MNSP1'].varValue,  flow['V_SA'].varValue  +flow['V_S_MNSP1'].varValue,\
    -flow['T_V_MNSP1'].varValue,-flow['VIC1_NSW1'].varValue-flow['V_SA'].varValue  -flow['V_S_MNSP1'].varValue +flow['T_V_MNSP1'].varValue]

    flows = {"INTERVAL_DATETIME": DI_current, "Terranora_flow": flow['N_Q_MNSP1'].value(),  "QNI_flow": flow['NSW1_QLD1'].value(),\
    "Basslink_flow":    flow['T_V_MNSP1'].value(), "Murraylink_flow":  flow['V_S_MNSP1'].value(), "Heywood_flow": flow['V_SA'].value(),\
    "VIC_NSW_flow":     flow['VIC1_NSW1'].value(), "Terranora_losses": losses['N_Q_MNSP1'].value(), "QNI_losses": losses['NSW1_QLD1'].value(),\
    "Basslink_losses":  losses['T_V_MNSP1'].value(),"Murraylink_losses":losses['V_S_MNSP1'].value(), "Heywood_losses": losses['V_SA'].value(),\
    "VIC_NSW_losses":   losses['VIC1_NSW1'].value()}
    

    g_e = []
    for region in regions:
        v1 = sum(v[b].varValue for b in s.index if s.loc[b, "REGIONID"] == region) 
        g_e.append(v1)

    load2 = []
    for region in regions:
        v2 = sum(l[b].varValue for b in load.index if load.loc[b, "REGIONID"] == region) 
        load2.append(v2)

    regional_losses = []
    for loss in range(5):
        regional_losses.append(round(g_e[loss] - demand_by_region[regions[loss]] + load2[loss] + interchange[loss],2))

    prices = []
    marginal = [{'name':name,'shadow price':c.pi,'slack':c.slack} for name, c in prob.constraints.items()]
    df_m = pd.DataFrame(marginal)
    #prices = (df_m.loc[1:5, "shadow price"] + df_m.loc[0, "shadow price"]).tolist()
    prices = (df_m.loc[0:4, "shadow price"]).tolist()
    
    df_latest = pd.DataFrame({'INTERVAL_DATETIME':DI_current,'REGIONID':regions,'Demand_MW':list(demand_by_region.values()),\
                              'Generation_MW':g_e,'Load_MW':load2,'Interchange_MW':interchange,'Modelled_RRP':prices,'Losses': regional_losses}) 
      #Positive (+) Net Interchange: The region is a net importer (electricity is flowing into the region). Negative (-) Net Interchange: The region is a net exporter (electricity is flowing out of the region). 
    return prob,v,l,df_latest,flows

def set_up_model(DI_current,df_pasa_duid_dict,bids_by_region,pasa_SC,df_duid_data,rerun_counter):
    
    current_time = (24*60 + DI_current.hour * 60 + DI_current.minute)
    df_pasa_duid_dict_now = df_pasa_duid_dict[DI_current]
    if st_pasa == True:
        #df_pasa_duid_dict_now = df_pasa_duid_dict[DI_current]
        adj_duids = ['Black_Coal','Brown_Coal','Gas','Battery','Hydro','Diesel']
    else:
        #df_pasa_duid_dict_now = df_pasa_duid_dict[DI_current.normalize()]
        adj_duids = ['Black_Coal','Brown_Coal'] #
        blk_coal_N,blk_coal_Q,brn_coal = 0,0,0 # remove ramping constraints
        
    df_duid_sched = df_pasa_duid_dict_now[~df_pasa_duid_dict_now["FUEL"].isin(["Wind", "Solar"])].copy() #need pasa here to avoid catching ss.
    
    results = []
    my_diffs = []
    for region in regions:
        df_region = bids_by_region[region] #select region
        df_region = df_region[df_region.TIME == current_time]
        target_sc = pasa_SC[(region, DI_current)] 
        df_region = df_region.copy()
        df_region['DELTA_SC'] = abs(df_region['SURPLUSCAPACITY'] - target_sc)
        df_region = df_region.sort_values(by=['DELTA_SC','DUID','PRICE'], ascending=[True,True,True])
        my_scarcity = df_region['DELTA_SC'].drop_duplicates().iloc[rerun_counter]

        #test3 = df_region['SURPLUSCAPACITY'].drop_duplicates().iloc[rerun_counter]
        #print("pasa target SC",target_sc,"bid data SC",test3)
        
        df_region = df_region[df_region.DELTA_SC == my_scarcity] 
        results.append(df_region)
        my_diffs+=[round(my_scarcity)]
    

    df_all_bids_all = pd.concat(results, ignore_index=True)

    coal_duids_in_bids = df_all_bids_all.loc[df_all_bids_all['FUEL'].isin(adj_duids),'DUID'].unique().tolist()
    #df_all_bids_all.to_csv('test.csv')
    coal_duids = df_duid_data.loc[df_duid_data['FUEL'].isin(adj_duids),'DUID']
    #print("bids:",len(coal_duids_in_bids),'pasa:',len(coal_duids))
    df_duid_sched_coal = df_duid_sched[df_duid_sched['DUID'].isin(coal_duids)]
    duid_sched_coal = list(df_duid_sched_coal.DUID)
    
    missing_in_bids = sorted(set(duid_sched_coal) - set(coal_duids_in_bids))#also get DUIDs in mtpasa but NOT in bids (sanity check)
    missing_in_pasa= sorted(set(coal_duids_in_bids) - set(duid_sched_coal ))
    
  
    print("SC_diff:",my_diffs,"added:",len(missing_in_bids),"removed:",len(missing_in_pasa), end=' ')
    #add in missing coal bids. Use the first timestamp in the full bid list
    df_filtered = df_all_bids[df_all_bids['DUID'].isin(missing_in_bids)]
    df_filtered = df_filtered.copy()
    df_filtered['DELTA_T'] = abs(current_time - df_filtered['TIME']) 
    df_filtered = df_filtered.sort_values(by=['DUID','DELTA_T','PRICE'], ascending=[True,True,True])
    df_filtered = df_filtered[df_filtered['INTERVAL_DATETIME'] == df_filtered.groupby('DUID')['INTERVAL_DATETIME'].transform('first')]
    df_all_bids_all = pd.concat([df_all_bids_all,df_filtered], ignore_index=True)
    #remove coal DUIDs not in PASA
    df_all_bids_all = df_all_bids_all[~df_all_bids_all['DUID'].isin(missing_in_pasa)]
    #tidy ups
    df_all_bids_all = df_all_bids_all.drop(columns=["SURPLUSCAPACITY", "TIME", "DELTA_SC","DELTA_T"]) 
    
    s = df_all_bids_all.loc[df_all_bids_all["GEN_LOAD"].eq("GEN")].copy() #generation
    s = s.reset_index(drop=True) # Create a stable bid id
    
    #bring in wind and solar with random +/1 $1 bids
    df_ss = df_pasa_duid_dict_now[df_pasa_duid_dict_now["FUEL"].isin(["Wind", "Solar"])].copy()
    df_ss = df_ss.rename(columns={"GENERATION_MAX_AVAILABILITY": "VOL"})
    df_pasa_duid_dict_now = df_pasa_duid_dict[DI_current]
    df_ss = df_ss[['INTERVAL_DATETIME','DUID','REGIONID','VOL','FUEL']]
 
    n = 7
    df_ss = df_ss.loc[df_ss.index.repeat(n)].copy()
    df_ss['VOL'] = df_ss['VOL'] / n
    df_ss = df_ss.reset_index(drop=True)
    
    df_ss['BAND'] = df_ss.groupby(level=0).cumcount() % n
    
    # Fixed — seed derived from the interval timestamp:
    rng = np.random.default_rng(seed=int(DI_current.timestamp()))
    df_ss['PRICE'] = rng.uniform(-1.5, 0.5, len(df_ss))   #df_ss['PRICE'] = np.random.uniform(-1.5, 0.5, len(df_ss))
    df_ss["GEN_LOAD"] = 'GEN'
  
    s = pd.concat([s,df_ss])
    s = s.sort_values(by=["PRICE"]).reset_index(drop=True)

    #prepare load table
    load = df_all_bids_all.loc[df_all_bids_all["GEN_LOAD"].eq("LOAD")].copy() #loads
    load["VOL"] = load["VOL"] * -1
    load = load.sort_values(by=["PRICE"],ascending=True).reset_index(drop=True) # Create a stable bid id
    
    return s,load


print("OK")


OK


In [4]:

#3.  this is the dispatch model
start = time.time()
print("Starting...")

st_pasa = True

if is_databricks() == True:
    base = Path('/Volumes/exploration/bills_repository/files/')
    df_duid_data = pd.read_parquet(base / "df_duid_LIVE.parquet")
    pulp.LpSolverDefault.msg = 0
    
    try: # need to remove them if they exist
        dbutils.widgets.remove("ref_year")
        dbutils.widgets.remove("st_pasa")
    except:
        pass
    
    dbutils.widgets.text("ref_year", "0") #defaults
    ref_year = int(dbutils.widgets.get("ref_year"))
    
    dbutils.widgets.text("st_pasa", "True") #defaults
    st_pasa = dbutils.widgets.get("st_pasa").strip().lower() == "true"
    
    if st_pasa == True:
        df_pasa = spark.table("exploration.bills_repository.pasa").toPandas()
        df_pasa_duid = spark.table("exploration.bills_repository.pasa_duid").toPandas()
        my_insert = []
    else:
        df_pasa = pd.read_parquet(base / "MTPASA_by_ref_years.parquet") 
        df_pasa_duid = pd.read_parquet(base / "MTPASA_duid_daily.parquet") 
        my_insert = ['REF_YEAR']
      
else:
    base = Path("C:/Users/BillNixey/OneDrive - Squadron Energy/Desktop/Working_files/New_model/test/")
    df_duid_data = pd.read_csv(base / "df_duid_LIVE.csv") 
    pulp.LpSolverDefault.msg = 1   
    
    if st_pasa == True:
        ref_year = 0
        df_pasa = pd.read_csv(base / "PASA.csv") 
        df_pasa_duid = pd.read_csv(base / "PASA_duid.csv") 
        my_insert = []
    else:
        ref_year = 2025
        df_pasa = pd.read_parquet(base / "MTPASA_by_ref_years.parquet")
        df_pasa_duid = pd.read_parquet(base / "MTPASA_duid_daily.parquet") 
        my_insert = ['REF_YEAR']
    

df_pasa_duid["INTERVAL_DATETIME"] = pd.to_datetime(df_pasa_duid["INTERVAL_DATETIME"])

df_all_bids = pd.read_parquet(base / "Bid_data_LIVE.parquet")
df_all_bids["FUEL"] = df_all_bids["FUEL"].str.replace(" ", "_", regex=False)

df_SC = pd.read_parquet(base / "spare_capacity_LIVE.parquet")
df_actual_ss = pd.read_parquet(base / "Actual_wind_solar_LIVE.parquet")
df_SC = pd.merge(df_SC,df_actual_ss, on=['SETTLEMENTDATE','REGIONID'],how = 'left')
df_SC = df_SC.rename(columns={"SETTLEMENTDATE": "INTERVAL_DATETIME"})
df_SC= df_SC.assign(SURPLUSCAPACITY = df_SC['Wind'] + df_SC['Solar']  - df_SC['TOTALDEMAND']  )
df_SC = df_SC[["INTERVAL_DATETIME","REGIONID","SURPLUSCAPACITY"]]

df_all_bids["INTERVAL_DATETIME"] = pd.to_datetime(df_all_bids["INTERVAL_DATETIME"])

df_all_bids = df_all_bids.sort_values(by=["INTERVAL_DATETIME", "PRICE"],ascending=True)
#add REGIONID and MLFs to bid file

df_pasa['SURPLUSCAPACITY'] =  df_pasa["SS_WIND_UIGF"] + df_pasa["SS_SOLAR_UIGF"] - df_pasa["DEMAND50"] #  df_pasa['AGGREGATECAPACITYAVAILABLE']+ df_pasa["SEMISCHEDULEDCAPACITY"] # this won't be needed once a new bid loader has been run, as SC will be calculated under the new approach#- df_pasa["AGGREGATESCHEDULEDLOAD"] 
df_pasa = df_pasa.rename(columns={"INTERVAL_DATETIME": "SETTLEMENTDATE","DEMAND50": "TOTALDEMAND"})
df_pasa["SETTLEMENTDATE"] = pd.to_datetime(df_pasa["SETTLEMENTDATE"])
df_pasa=df_pasa.sort_values(by=my_insert+["SETTLEMENTDATE","REGIONID"],ascending=True).reset_index(drop=True)
df_pasa = df_pasa[['SETTLEMENTDATE','SURPLUSCAPACITY','TOTALDEMAND','REGIONID','SS_WIND_UIGF','SS_SOLAR_UIGF']+my_insert]#,'RRP'

my_DIs = df_pasa["SETTLEMENTDATE"].drop_duplicates().sort_values().tolist() #

if st_pasa == True:
    #safety check to ensure both pasa files have the same timestamp
    t = list(df_pasa_duid.INTERVAL_DATETIME) 
    t2 = list(dict.fromkeys(t))
    my_DIs = [x for x in my_DIs if x in t2]
else:
    print("Weather reference year:",ref_year)
    print("mt pasa region range:",df_pasa["SETTLEMENTDATE"].min(),df_pasa["SETTLEMENTDATE"].max())
    print("mt pasa duid range:",df_pasa_duid["INTERVAL_DATETIME"].min(),df_pasa_duid["INTERVAL_DATETIME"].max())
    df_pasa = df_pasa[df_pasa.REF_YEAR == ref_year]
    df_pasa_duid = df_pasa_duid[(df_pasa_duid.REF_YEAR == ref_year) | (df_pasa_duid.REF_YEAR == 0)]
    

if is_databricks() == False:
    my_DIs = my_DIs[0:5] ###################################################
else:
    pass



#add histrocial SC
df_all_bids = pd.merge(df_all_bids,df_SC,on=['INTERVAL_DATETIME','REGIONID'],how='left') ######### do I need to merge here?
df_all_bids['TIME'] = 24*60+df_all_bids['INTERVAL_DATETIME'].dt.hour*60+df_all_bids['INTERVAL_DATETIME'].dt.minute #add a day of miuntes to avoid negatives

#split to region
bids_by_region = {r: df for r, df in df_all_bids.groupby("REGIONID")}


#target SCs
pasa_SC = (df_pasa.set_index(["REGIONID", "SETTLEMENTDATE"])["SURPLUSCAPACITY"].to_dict())
#DUIDs by interval
df_pasa_duid_dict = {f: df for f, df in df_pasa_duid.groupby("INTERVAL_DATETIME")}

# --- Results holders


regional_parts = []
fuel_parts = []
constraint_parts = []
sqe_parts = []
flows_parts = []

#########################################

for DI_current in my_DIs:
    
    print(my_DIs.index(DI_current),end=' ')# counter
    rerun_counter = 0
    
    s,load = set_up_model(DI_current,df_pasa_duid_dict,bids_by_region,pasa_SC,df_duid_data,rerun_counter)
    prob,v,l,df_latest,flows = prob_solve(DI_current,s,load) 
    print(DI_current,f"${df_latest.Modelled_RRP.mean():,.2f}","/MWh")
    
    while ((df_latest.Modelled_RRP[0] + df_latest.Modelled_RRP[4]) > 2000) and (rerun_counter <4) : #if the addition of NSW and VIC prices are greater than $1000, resolve 
        rerun_counter += 1
        print('Re-run',rerun_counter)
        s,load = set_up_model(DI_current,df_pasa_duid_dict,bids_by_region,pasa_SC,df_duid_data,rerun_counter)
        prob,v,l,df_latest,flows = prob_solve(DI_current,s,load)

    #create and save combined load and gen stack
    v_out = {bid_id: v[bid_id].value() for bid_id in v}
    energy_df = (pd.Series(v_out, name="ENERGY").reset_index())
    df_s = pd.concat([s,energy_df],axis=1)
    l_out = {bid_id: l[bid_id].value() for bid_id in l}
    load_df = (pd.Series(l_out, name="ENERGY").reset_index())
    df_l = pd.concat([load,load_df],axis=1)
    df_s = pd.concat([df_s,df_l])
    df_s = df_s.sort_values(by=["PRICE"],ascending=True).reset_index(drop=True)
    df_s['INTERVAL_DATETIME'] = DI_current
    #ensure battery charge and discharge treated separately
    df_s['FUEL_SPLIT'] = np.where(df_s['FUEL'].eq('Battery') & df_s['ENERGY'].lt(0), 'Battery_load',
    np.where(df_s['FUEL'].eq('Battery') & df_s['ENERGY'].ge(0), 'Battery_gen', df_s['FUEL']))
    out_energy = (df_s.groupby(['INTERVAL_DATETIME', 'FUEL_SPLIT','REGIONID'])['ENERGY'].sum().unstack('FUEL_SPLIT', fill_value=0))
    #get regional wind solar curtailment
    out_vol = (df_s.groupby(['INTERVAL_DATETIME', 'FUEL_SPLIT','REGIONID'])['VOL'].sum().unstack('FUEL_SPLIT', fill_value=0))
   
    if 'Solar' in out_vol.columns:
        pass
    else:
        out_energy['Solar'] = 0
        out_vol['Solar'] = 0
    out_vol = out_vol.reset_index()
    out_vol = out_vol[['INTERVAL_DATETIME','REGIONID','Solar','Wind']]
    out_vol = out_vol.rename(columns={'Solar': 'Solar_UIGF','Wind': 'Wind_UIGF'})
    out_energy = pd.merge(out_energy,out_vol,on=['INTERVAL_DATETIME','REGIONID'],how='left')
    out_energy['Solar_curtailment'] = (out_energy['Solar_UIGF'] - out_energy['Solar']).round(2)
    out_energy['Wind_curtailment'] = (out_energy['Wind_UIGF'] - out_energy['Wind']).round(2)

    #monitor battery levels on a regional basis
    for c in ['Battery_gen', 'Battery_load']:
        if c not in out_energy.columns:
            out_energy[c] = 0
    if DI_current == my_DIs[0]:
        out_energy['Energy_storage'] = -out_energy['Battery_gen']/12 - out_energy['Battery_load']*.82/12 
    else:
        out_energy['Energy_storage'] = - out_energy['Battery_load'] *.82/12  - out_energy['Battery_gen']/12 + fuel_parts[-1]['Energy_storage'].values
    
    # for ramping
    blk_coal_N = out_energy.loc[out_energy["REGIONID"] == "NSW1", "Black_Coal"].sum()
    blk_coal_Q = out_energy.loc[out_energy["REGIONID"] == "QLD1", "Black_Coal"].sum()
    brn_coal = out_energy["Brown_Coal"].sum()

    df_SQE_now = df_s[df_s.DUID.isin(SQE_duid)]
   
    regional_parts.append(df_latest)
    fuel_parts.append(out_energy)
    constraint_parts.append(get_constraints(DI_current,prob.constraints.items()))
    sqe_parts.append(df_SQE_now)
    flows_parts.append(flows)
    
    

df_regional = pd.concat(regional_parts, ignore_index=True)
df_fuel = pd.concat(fuel_parts, ignore_index=True)
df_constraints = pd.concat(constraint_parts, ignore_index=True)
df_SQE = pd.concat(sqe_parts, ignore_index=True)
df_flows = pd.DataFrame(flows_parts)

if st_pasa == True:
    pass
else:
    ref_year = df_pasa.loc[df_pasa["SETTLEMENTDATE"] == DI_current,"REF_YEAR"].iloc[0]
    flows["REF_YEAR"] = ref_year #a dict
    df_constraints['REF_YEAR'] = ref_year
    df_fuel['REF_YEAR'] = ref_year
    df_regional['REF_YEAR'] = ref_year
    df_SQE['REF_YEAR'] = ref_year


df_SQE = pd.DataFrame(df_SQE).reset_index(drop=True)
df_SQE['CURTAILMENT'] =  df_SQE['VOL'] - df_SQE['ENERGY']
df_SQE = df_SQE[['INTERVAL_DATETIME','DUID','VOL','REGIONID','FUEL','ENERGY','CURTAILMENT']+my_insert]
df_SQE = (df_SQE.groupby(['INTERVAL_DATETIME', 'DUID', 'REGIONID', 'FUEL']+my_insert,as_index=False)[['VOL', 'ENERGY', 'CURTAILMENT']].sum())
df_SQE[['VOL', 'ENERGY', 'CURTAILMENT']] = (df_SQE[['VOL', 'ENERGY', 'CURTAILMENT']].round(0))


df_fuel = df_fuel.fillna(0).reset_index(drop=False)




if is_databricks() == True:

    if st_pasa == True:
        
        spark.conf.set("spark.sql.session.timeZone", "Australia/Brisbane")
            
        df_fuel['INTERVAL_DATETIME'] = (pd.to_datetime(df_fuel['INTERVAL_DATETIME']).dt.tz_localize("Australia/Brisbane").dt.tz_localize(None))
        df_spark = spark.createDataFrame(df_fuel)
        df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("exploration.bills_repository.FUEL")
    
        #df_regional = df_regional[df_regional.REGIONID.isin(['NSW1','QLD1','VIC1','SA1'])]
        df_regional = pd.merge(df_regional,df_pp,on=['INTERVAL_DATETIME','REGIONID'],how='left')
        df_regional['INTERVAL_DATETIME'] = (pd.to_datetime(df_regional['INTERVAL_DATETIME']).dt.tz_localize("Australia/Brisbane").dt.tz_localize(None))
        df_spark = spark.createDataFrame(df_regional.reset_index(drop=True))
        df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("exploration.bills_repository.REGIONAL")

        now = datetime.now(ZoneInfo("Australia/Brisbane"))
        s1 = str(now)
        s2 = s1[:10]
        s3 = s2.replace('-', '')
        s4 = s1[11:13]
        df_regional.reset_index(drop=True).to_csv(base / 'backcast' /f'df_regional_{s3+"H"+s4}.csv',index=False)
        
        
        df_flows['INTERVAL_DATETIME'] = (pd.to_datetime(df_flows['INTERVAL_DATETIME']).dt.tz_localize("Australia/Brisbane").dt.tz_localize(None))
        df_spark = spark.createDataFrame(df_flows.reset_index(drop=True))
        df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("exploration.bills_repository.FLOWS")
    
        df_SQE['INTERVAL_DATETIME'] = (pd.to_datetime(df_SQE['INTERVAL_DATETIME']).dt.tz_localize("Australia/Brisbane").dt.tz_localize(None))
        df_spark = spark.createDataFrame(df_SQE.reset_index(drop=True))
        df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("exploration.bills_repository.SQE")

        df_constraints = df_constraints[~df_constraints["Constraint"].str.contains("energy|loss|ramp", case=False, na=False)]
        df_constraints['INTERVAL_DATETIME'] = (pd.to_datetime(df_constraints['INTERVAL_DATETIME']).dt.tz_localize("Australia/Brisbane").dt.tz_localize(None))
        df_spark = spark.createDataFrame(df_constraints.reset_index(drop=True))
        df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("exploration.bills_repository.CONSTRAINTS")
    
    else:
        df_regional.reset_index(drop=True).to_csv(base / 'mt_pasa' /f'df_regional_{str(ref_year)}.csv',index=False)
        df_SQE.reset_index(drop=True).to_csv(base / 'mt_pasa' /f'df_SQE_{str(ref_year)}.csv',index=False)
        df_fuel.reset_index(drop=True).to_csv(base / 'mt_pasa' /'output' /f'df_fuel_{str(ref_year)}.csv',index=False)
        df_constraints.reset_index(drop=True).to_csv(base / 'mt_pasa' /'output' /f'df_constraint_{str(ref_year)}.csv',index=False)

else:   
    df_flows.reset_index(drop=True).to_csv(base /f'df_flows_{str(ref_year)}.csv',index=False)
    df_constraints.reset_index(drop=True).to_csv(base /f'df_constraint_{str(ref_year)}.csv',index=False)
    df_fuel.reset_index(drop=True).to_csv(base /f'df_fuel_{str(ref_year)}.csv',index=False)
    df_regional.reset_index(drop=True).to_csv(base /f'df_regional_{str(ref_year)}.csv',index=False)
    df_SQE.reset_index(drop=True).to_csv(base /f'df_SQE_{str(ref_year)}.csv',index=False)
    prob.writeLP(base /"BidOpt.lp")
    df_s.reset_index(drop=True).to_csv(base /'df_stack.csv',index=False)



#convert pandas to spark pandas in Databricks
#Battery_load isn't labelled correctly in the bid stack. Dispatched volume (load) is OK. 


print("Done in", round(time.time() - start, 2), "seconds")
#df_fuel.head()
(df_regional.groupby("REGIONID")[["Demand_MW", "Generation_MW", "Load_MW","Interchange_MW", "Modelled_RRP", "Losses"]].mean().reset_index())

Starting...
0 SC_diff: [0, 23, 14, 3, 20] added: 16 removed: 31 2026-08-04 10:30:00 Optimal 2026-08-04 10:30:00 $9.54 /MWh
1 SC_diff: [10, 18, 3, 0, 113] added: 12 removed: 26 2026-08-04 11:00:00 Optimal 2026-08-04 11:00:00 $19.78 /MWh
2 SC_diff: [77, 57, 2, 3, 7] added: 18 removed: 20 2026-08-04 11:30:00 Optimal 2026-08-04 11:30:00 $24.15 /MWh
3 SC_diff: [15, 44, 63, 5, 26] added: 12 removed: 11 2026-08-04 12:00:00 Optimal 2026-08-04 12:00:00 $31.79 /MWh
4 SC_diff: [125, 19, 41, 3, 8] added: 18 removed: 15 2026-08-04 12:30:00 Optimal 2026-08-04 12:30:00 $0.36 /MWh
Done in 4.88 seconds


,REGIONID,Demand_MW,Generation_MW,Load_MW,Interchange_MW,Modelled_RRP,Losses
0,NSW1,6876.8,7076.373273,-1133.549996,997.016750,16.664390,63.040
1,QLD1,4177.8,6242.491338,-791.547667,-1196.905740,1.515556,76.240
2,SA1,1521.0,1169.160918,-327.371492,695.050186,32.904523,15.842
3,TAS1,1177.4,890.759960,0.000000,303.611262,18.402169,16.972
4,VIC1,6729.2,7837.202206,-290.400000,-798.772458,16.137017,18.828


[{'INTERVAL_DATETIME': Timestamp('2026-08-04 10:30:00'),
  'Terranora_flow': -107.0,
  'QNI_flow': -1135.5789,
  'Basslink_flow': -191.86133,
  'Murraylink_flow': 105.88639,
  'Heywood_flow': 502.287,
  'VIC_NSW_flow': -157.84855,
  'Terranora_losses': 12.395308,
  'QNI_losses': 125.0,
  'Basslink_losses': 11.030333,
  'Murraylink_losses': 4.7684073,
  'Heywood_losses': 16.924857,
  'VIC_NSW_losses': 1.6287979},
 {'INTERVAL_DATETIME': Timestamp('2026-08-04 11:00:00'),
  'Terranora_flow': -107.0,
  'QNI_flow': -1107.497,
  'Basslink_flow': -400.0,
  'Murraylink_flow': 105.88639,
  'Heywood_flow': 589.18884,
  'VIC_NSW_flow': -21.516992,
  'Terranora_losses': 12.395308,
  'QNI_losses': 125.0,
  'Basslink_losses': 22.1968,
  'Murraylink_losses': 4.7684073,
  'Heywood_losses': 25.442766,
  'VIC_NSW_losses': 0.16436236},
 {'INTERVAL_DATETIME': Timestamp('2026-08-04 11:30:00'),
  'Terranora_flow': -107.0,
  'QNI_flow': -1081.7878,
  'Basslink_flow': -324.21739,
  'Murraylink_flow': 220.0,
  

In [6]:
  '''
    prices = [-0.6, -0.2, 0.2, 0.6]
    mask = df_ss['DUID'].isin(SQE_duid)
    df_ss.loc[mask, 'PRICE'] = np.tile(prices,len(df_ss.loc[mask]) // n)
    df_ss["GEN_LOAD"] = 'GEN'
    #df_ss.to_csv(base/ 'test.csv')
    '''


'\n  prices = [-0.6, -0.2, 0.2, 0.6]\n  mask = df_ss[\'DUID\'].isin(SQE_duid)\n  df_ss.loc[mask, \'PRICE\'] = np.tile(prices,len(df_ss.loc[mask]) // n)\n  df_ss["GEN_LOAD"] = \'GEN\'\n  #df_ss.to_csv(base/ \'test.csv\')\n  '

In [7]:
 '''
    sqe_mask = df_ss['DUID'].isin(SQE_duid)

    df_sqe = df_ss[sqe_mask].copy()
    df_other = df_ss[~sqe_mask].copy()
    
    # Expand SQE to 4 bands
    df_sqe = df_sqe.loc[df_sqe.index.repeat(4)].reset_index(drop=True)
    df_sqe['VOL'] /= 4
    df_sqe['BAND'] = df_sqe.groupby(df_sqe.index // 4).cumcount()
    df_sqe['PRICE'] = np.tile([-0.6, -0.2, 0.2, 0.6], len(df_sqe) // 4)
    # Expand non-SQE to 7 bands
    df_other = df_other.loc[df_other.index.repeat(7)].reset_index(drop=True)
    df_other['VOL'] /= 7
    df_other['BAND'] = df_other.groupby(df_other.index // 7).cumcount()
    df_other['PRICE'] = np.random.uniform(-1, 1, len(df_other))
    
    # Recombine
    df_ss = pd.concat([df_sqe, df_other], ignore_index=True)
    
    df_ss['GEN_LOAD'] = 'GEN'
    '''  

"\n   sqe_mask = df_ss['DUID'].isin(SQE_duid)\n\n   df_sqe = df_ss[sqe_mask].copy()\n   df_other = df_ss[~sqe_mask].copy()\n   \n   # Expand SQE to 4 bands\n   df_sqe = df_sqe.loc[df_sqe.index.repeat(4)].reset_index(drop=True)\n   df_sqe['VOL'] /= 4\n   df_sqe['BAND'] = df_sqe.groupby(df_sqe.index // 4).cumcount()\n   df_sqe['PRICE'] = np.tile([-0.6, -0.2, 0.2, 0.6], len(df_sqe) // 4)\n   # Expand non-SQE to 7 bands\n   df_other = df_other.loc[df_other.index.repeat(7)].reset_index(drop=True)\n   df_other['VOL'] /= 7\n   df_other['BAND'] = df_other.groupby(df_other.index // 7).cumcount()\n   df_other['PRICE'] = np.random.uniform(-1, 1, len(df_other))\n   \n   # Recombine\n   df_ss = pd.concat([df_sqe, df_other], ignore_index=True)\n   \n   df_ss['GEN_LOAD'] = 'GEN'\n   "

In [8]:
'''
    M = 1000000

    for ic in IC_list:
        lf = IC_DATA[ic]['loss_factor']
    
        prob += losses[ic] - lf * flow[ic] <= M * (1 - direction[ic])
        prob += losses[ic] - lf * flow[ic] >= -M * (1 - direction[ic])
        
        prob += losses[ic] + lf * flow[ic] <= M * direction[ic]
        prob += losses[ic] + lf * flow[ic] >= -M * direction[ic]

    '''

"\n    M = 1000000\n\n    for ic in IC_list:\n        lf = IC_DATA[ic]['loss_factor']\n    \n        prob += losses[ic] - lf * flow[ic] <= M * (1 - direction[ic])\n        prob += losses[ic] - lf * flow[ic] >= -M * (1 - direction[ic])\n        \n        prob += losses[ic] + lf * flow[ic] <= M * direction[ic]\n        prob += losses[ic] + lf * flow[ic] >= -M * direction[ic]\n\n    "

In [9]:
  '''   
    #SET THE CONSTRAINTS 
    for region in regions:
       
        lhs = (pulp.lpSum(v[c] for c in s.index if s.loc[c, "REGIONID"] == region) \
               + pulp.lpSum(l[d] for d in load.index if load.loc[d, "REGIONID"] == region)) 
    
        if region == "NSW1":
            rhs = (flow['N_Q_MNSP1'] + flow['NSW1_QLD1'] - flow['VIC1_NSW1']) + 0.4394*losses['NSW1_QLD1']+0.5028*losses['N_Q_MNSP1']+(1-0.6262)*losses['VIC1_NSW1'] 
        elif region == "QLD1":
            rhs = (- flow['N_Q_MNSP1'] - flow['NSW1_QLD1']) +(1-0.4394)*losses['NSW1_QLD1']+(1-0.5028)*losses['N_Q_MNSP1']
        elif region == "SA1":
            rhs = (- flow['V_S_MNSP1'] - flow['V_SA']) + (1-0.4925)*losses['V_SA']+(1-0.5057)*losses['V_S_MNSP1']
        elif region == "TAS1":
            rhs = (flow['T_V_MNSP1'])  +  losses['T_V_MNSP1']
        else:
            rhs = (flow['VIC1_NSW1'] + flow['V_SA'] + flow['V_S_MNSP1'] - flow['T_V_MNSP1']) + 0.4925*losses['V_SA']+0.6262*losses['VIC1_NSW1']+0.5057*losses['V_S_MNSP1']
    
        prob += (lhs == demand_by_region[region] + rhs, f"{region}_energy_balance")
  
    #ramping
    if blk_coal_Q >0:
        prob += pulp.lpSum(v[b] for b in s.index if (s.loc[b, "FUEL"] == "Black_Coal") & (s.loc[b, "REGIONID"] == "QLD1")) <= 1.18 * blk_coal_Q,"Black_coal_ramping_up_Q"
        prob += pulp.lpSum(v[b] for b in s.index if (s.loc[b, "FUEL"] == "Black_Coal") & (s.loc[b, "REGIONID"] == "QLD1")) >= 0.82 * blk_coal_Q,"Black_coal_ramping_down_Q"
    else:
        pass
    if blk_coal_N >0:    
        prob += pulp.lpSum(v[b] for b in s.index if (s.loc[b, "FUEL"] == "Black_Coal") & (s.loc[b, "REGIONID"] == "NSW1")) <= 1.18 * blk_coal_N,"Black_coal_ramping_up_N"
        prob += pulp.lpSum(v[b] for b in s.index if (s.loc[b, "FUEL"] == "Black_Coal") & (s.loc[b, "REGIONID"] == "NSW1")) >= 0.82 * blk_coal_N,"Black_coal_ramping_down_N"
    else:
        pass
    if brn_coal >0:
        prob += pulp.lpSum(v[b] for b in s.index if s.loc[b, "FUEL"] == "Brown_Coal") <= 1.18 * brn_coal,"Brown_coal_ramping_up"
        prob += pulp.lpSum(v[b] for b in s.index if s.loc[b, "FUEL"] == "Brown_Coal") >= 0.82 * brn_coal,"Brown_coal_ramping_down"
    else:
        pass



    
    #top 4 binding thermal constraints  in April 2026
    N_94T_947_72 = {"FLYCRKWF": -1.0,"NYNGAN1": 0.616, "NEVERSF1": 0.616, "WELLSF1": 0.616, "SUNTPSF1": 0.598,"MOLNGSF1": 0.554,"BODWF1":0.544,
                    "MANSLR1":0.531,"PARSF1":0.457,"GOONSF1":0.457,"QPSFB1":0.457,"QPSFB2":0.457,"WELNSF1":0.439,"ORABESS1":0.439,"BERYLSF1":0.41,
                   "JEMALNG1":0.387,"STUBSF1":0.271,"STUBSF2":0.271,"WOLARSF1":0.153}
    prob += pulp.lpSum(N_94T_947_72.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) <= 475,"N>94T_947_72"
    

    N_NIL_94T =  {"MOLNGSF": 1.0,"MANSLR1": 0.873,"PARSF1": 0.475,"GOONSF1": 0.475,"QPSFB1": 0.475,"QPSFB2": 0.475 ,
                  "FLYCRKWF": -0.446,"JEMALNG1":0.402 ,"SUNTPSF":0.189 ,"NYNGAN1": 0.153,"NEVERSF1": 0.153,"WELLSF1": 0.153}
    prob += pulp.lpSum(N_NIL_94T.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) <= 180,"N>NIL_94T"
   
    
    N_NIL_969 = {"GNNDHSF1": 1.0,"MOREESF1": 0.301}
    prob += pulp.lpSum(N_NIL_969.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) <= 110,"N>NIL_969"    
    
    N_NIL_060_051 = {'GESF1': +1,	'HUMENSW': +0.995,	'WLWLSF2': +0.995,	'WLWLSF1': +0.995,	'CUSF1': +0.992,	'CRWASF1': +0.976,\
                     'MULWASF1': +0.963,	'BLOWERNG': +0.947,	'WAGGNSF1': +0.947,	'JUNEESF1': +0.947,	'SEBSF1': +0.947,	'WSTWYSF1': +0.947,\
                     'WYASF1': +0.947,	'BOMENSF1': +0.947,	'FINLYSF1': +0.945,	'URANQ11': +0.944,	'URANQ12': +0.944,	'URANQ13': +0.944,\
                     'URANQ14': +0.944,	'AVLSF1': +0.917,	'COLEASF1': +0.883,	'HILLSTN1': +0.879,	'DARLSF1': +0.879,	'RIVNB2': +0.879,\
                     'RESS1': +0.879,	'DPNTB1': +0.879,	'LIMOSF21': +0.459,	'LIMOSF11': +0.459,	'SUNRSF1': +0.459,	'LIMBESS1': +0.459,\
                     'BROKENH1': +0.248,	'BHB1': +0.248,	'STWF1': +0.248,	'KARSF1': +0.207,	'YATSF1': +0.207,	'BANN1': +0.178,\
                     'WEMENSF1': +0.178,	'KIAMSF1': +0.172}
    prob += pulp.lpSum(N_NIL_060_051.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) - 0.207* flow['V_S_MNSP1'] <= 1300,"N>>NIL_060_051"

    #consider Q_STR_7C0K_MEWF_16,  N>9GL_999_998,N>>BDBU_060_051, S>NIL_HUWT_STBG
    
    #VIC-NSW limits
    #transient stability
    V_N_NIL_V2  =  {'DARTM1': -0.896,	'MCKAY1': -0.896,	'WKIEWA1': -0.896,	'WKIEWA2': -0.896,	'MURRAY': -0.786,	'ARWF1': -0.432,\
                    'BALB1': -0.432,	'YENDWF1': -0.432,	'BRYB1WF1': -0.432,	'BRYB2WF2': -0.432,	'BULGANA1': -0.432,	'BULBES1': -0.432,\
                    'RANGEB1': -0.432,	'CROWLWF1': -0.432,	'MERCER01': -0.432,	'MOORAWF1': -0.432,	'ELAINWF1': -0.432,	'GLENSF1': -0.432,\
                    'MOKOSF1': -0.432,	'GLRWNSF1': -0.432,	'WINTSF1': -0.432,	'MTGELWF1': -0.432,	'KIATAWF1': -0.432,	'HBESS1': -0.432,\
                    'GANNB1': -0.432,	'KERNGSP1': -0.432,	'GANNSF1': -0.432,	'COHUNSF1': -0.432,	'KIAMSF1': -0.432,	'KESSB1': -0.432,\
                    'VBB1': -0.432,	'MUWAWF1': -0.432,	'MUWAWF2': -0.432,	'PIBESS1': -0.432,	'BALDHWF1': -0.432,	'KARSF1': -0.432,\
                    'YATSF1': -0.432,	'NUMURSF1': -0.432,	'GIRGSF': -0.432,	'WUNUSF1': -0.432,	'LANCSF1': -0.432,	'CHYTWF1': -0.432,\
                    'MRNBESS1': -0.432,	'MRTLSWF1': -0.432,	'TRGBESS1': -0.432,	'SALTCRK1': -0.432,	'OAKLAND1': -0.432,	'HD1WF1': -0.432,\
                    'RYANCWF1': -0.432,	'MACARTH1': -0.432,	'BANN1': -0.432,	'WEMENSF1': -0.432,	'YWPS1': -0.36,	'YWPS2': -0.36,	'YWPS3': -0.36,\
                    'YWPS4': -0.36}
    prob += pulp.lpSum(V_N_NIL_V2.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) + 1 * flow['VIC1_NSW1'] + 0.492* flow['V_S_MNSP1'] \
    +0.478 *flow['V_SA'] -0.398 * flow['T_V_MNSP1']  <= 0,"V::N_NIL_V2"
    #voltage
    V_N_NIL_V1 = {'ALDGASF1': +1,	'BBATTERY1': +0.485,	'KIDSPHG1': +0.452,	'KIDSPHG2': +0.452,	'KIDSPHL1': -0.452,	'KIDSPHL2': -0.452,\
                  'BARRON-1': +0.452,	'BARRON-2': +0.452,	'DAYDSF1': +0.452,	'HAYMSF1': +0.452,	'YABULU2': +0.452,	'HAUGHT11': +0.452,\
                  'KAREEYA1': +0.452,	'KAREEYA2': +0.452,	'KAREEYA3': +0.452,	'KAREEYA4': +0.452,	'MSTUART1': +0.452,	'MSTUART2': +0.452,\
                  'MSTUART3': +0.452,	'KSP1': +0.452,	'RRSF1': +0.452,	'KEPWF1': +0.452,	'KEPSF1': +0.452,	'KABANWF1': +0.452,	\
                  'YABULU': +0.452,	'SMCSF1': +0.452,	'MEWF1': +0.452,	'CLARESF1': +0.451,	'CSPVPS1': +0.451,	'HAMISF1': +0.451,\
                  'WHITSF1': +0.451,	'BRDDSF01': +0.45,	'BRDDBES1': +0.45,	'CLRKCWF1': +0.45,	'CLRKCWF2': +0.45,	'RUGBYR1': +0.448,\
                  'Stanwell': +0.443,	'STAN-1': +0.443,	'STAN-2': +0.443,	'STAN-3': +0.443,	'STAN-4': +0.443,	'BARCALDN': +0.434,\
                  'LILYSF1': +0.434,	'CLERMSF1': +0.434,	'MIDDLSF1': +0.433,	'EMERASF1': +0.426,	'MOUSF1': +0.29,	'GSTONE1': -0.27,\
                  'GSTONE2': -0.27,	'GSTONE5': -0.27,	'GSTONE6': -0.27,	'CALL_B_1': +0.257,	'CALL_B_2': +0.257,	'CPP_3': +0.257,\
                  'CPP_4': +0.257,	'GSTONE3': -0.243,	'GSTONE4': -0.243,	'BUSF1': -0.172,	'CHILDSF1': -0.153}
    prob += pulp.lpSum(V_N_NIL_V1.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) + 1 * flow['VIC1_NSW1'] \
    + 0.192* flow['V_S_MNSP1']   <= 1150,"V^^N_NIL_V1"
    
    #NSW_QLD limits & Sapphire
    Q_N_NIL_SRAR = {'SAPHWF1': -1.04}
    prob += pulp.lpSum(Q_N_NIL_SRAR.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) - 1*flow['NSW1_QLD1'] <= 1200,"Q^^N_NIL_SRAR"

    #Murra Warra 1 and 2
    V_NIL_MLGT_MLGT = {'VBB1': +1,	'MERCER01': +0.932,	'MOORAWF1': +0.932,	'ELAINWF1': +0.932,	'BRYB1WF1': +0.869,	'BRYB2WF2': +0.869,	'MRTLSWF1': +0.838,	'TRGBESS1': +0.838,	'ARWF1': +0.837,	'CROWLWF1': +0.829,	'BALB1': +0.824,	'YENDWF1': +0.822,	'BULGANA1': +0.815,	'BULBES1': +0.815,	'KIATAWF1': +0.76,	'MTGELWF1': -0.735,	'MUWAWF1': +0.726,	'MUWAWF2': +0.726,	'SALTCRK1': +0.685,	'OAKLAND1': +0.685,	'KIAMSF1': +0.598,	'GANNB1': +0.578,	'KERNGSP1': +0.578,	'GANNSF1': +0.578,	'COHUNSF1': +0.578,	'KESSB1': +0.575,	'LNGS1': -0.564,	'LNGS2': -0.564,	'CRWARP1': +0.548,	'BANN1': +0.542,	'WEMENSF1': +0.542,	'KARSF1': +0.529,	'YATSF1': +0.529,	'HD1WF1': +0.384,	'RYANCWF1': +0.384,	'MACARTH1': +0.384,	'MLB01': +0.382,	'DUNDWF1': +0.382,	'DUNDWF2': +0.382,	'DUNDWF3': +0.382,	'MORTLK11': +0.382,	'MORTLK12': +0.382,	'STOCKYD1': +0.381,	'GPWFWST1': +0.38,	'GPWFWST2': +0.38,	'GPWFEST1': +0.38,	'GPWFEST2': +0.38,	'GPWFEST3': +0.38,	'BROKENH1': +0.358,	'BHB1': +0.358,	'STWF1': +0.358,	'NUMURSF1': +0.278,	'GIRGSF': +0.278,	'WUNUSF1': +0.278,	'LANCSF1': +0.278,	'LIMOSF21': +0.255,	'LIMOSF11': +0.255,	'SUNRSF1': +0.255,	'LIMBESS1': +0.255,	'GOESF1': +0.24,	'GLENSF1': +0.224,	'MOKOSF1': +0.22,	'GLRWNSF1': +0.22,	'WINTSF1': +0.22,	'NPS': -0.186,	'MREHA2': +0.155,	'MREHA3': +0.155,	'MREHA1': +0.155,	'MURRAY': +0.153,	'HUMEV': +0.151}
    prob += pulp.lpSum(V_NIL_MLGT_MLGT.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) - 0.159 * flow['VIC1_NSW1'] - 0.529 * flow['V_S_MNSP1'] - 0.433 * flow['V_SA'] <= 2200,"V>>NIL_MLGT_MLGT"

    V_NWVIC_GFT1_750 = {'ARWF1':1,'BULGANA1':1,'BULBES1':1,'CROWLWF1':1,'MUWAWF1':1,'MUWAWF2':1}
    prob += pulp.lpSum(V_NWVIC_GFT1_750.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) <= 500,"V_NWVIC_GFT1_750"
    
    #Bango
    N_N_NIL_WGLT = {'HILLSTN1': +1,	'DARLSF1': +1,	'RIVNB2': +1,	'RESS1': +1,	'DPNTB1': +1,	'COLEASF1': +0.994,	'AVLSF1': +0.985,	'URANQ11': +0.972,	'URANQ12': +0.972,	'URANQ13': +0.972,	'URANQ14': +0.972,	'WAGGNSF1': +0.939,	'JUNEESF1': +0.939,	'SEBSF1': +0.939,	'WSTWYSF1': +0.939,	'WYASF1': +0.939,	'BOMENSF1': +0.939,	'FINLYSF1': +0.898,	'MULWASF1': +0.846,	'CUSF1': +0.812,	'CRWASF1': +0.81,	'WLWLSF2': +0.793,	'WLWLSF1': +0.793,	'HUMENSW': +0.756,	'GESF1': +0.736,	'LIMOSF21': +0.698,	'LIMOSF11': +0.698,	'SUNRSF1': +0.698,	'LIMBESS1': +0.698,	'BLOWERNG': +0.602,	'BROKENH1': +0.54,	'BHB1': +0.54,	'STWF1': +0.54,	'MURRAY': -0.291,	'GUNNING1': +0.216,	'BANGOWF1': +0.202,	'BANGOWF2': +0.202,	'KARSF1': +0.156,	'YATSF1': +0.156}
    prob += pulp.lpSum(N_N_NIL_WGLT.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index) + 0.376 * flow['VIC1_NSW1'] - 0.156* flow['V_S_MNSP1']   <= 1050,"N^^N_NIL_WGLT"

    #Clarke Creek 1 and 2
    Q_NIL_LCCP_BCCP = {'ALDGASF1': +1,	'BBATTERY1': +0.485,	'KIDSPHG1': +0.452,	'KIDSPHG2': +0.452,	'KIDSPHL1': -0.452,	'KIDSPHL2': -0.452,	'BARRON-1': +0.452,	'BARRON-2': +0.452,	'DAYDSF1': +0.452,	'HAYMSF1': +0.452,	'YABULU2': +0.452,	'HAUGHT11': +0.452,	'KAREEYA1': +0.452,	'KAREEYA2': +0.452,	'KAREEYA3': +0.452,	'KAREEYA4': +0.452,	'MSTUART1': +0.452,	'MSTUART2': +0.452,	'MSTUART3': +0.452,	'KSP1': +0.452,	'RRSF1': +0.452,	'KEPWF1': +0.452,	'KEPSF1': +0.452,	'KABANWF1': +0.452,	'YABULU': +0.452,	'SMCSF1': +0.452,	'MEWF1': +0.452,	'CLARESF1': +0.451,	'CSPVPS1': +0.451,	'HAMISF1': +0.451,	'WHITSF1': +0.451,	'BRDDSF01': +0.45,	'BRDDBES1': +0.45,	'CLRKCWF1': +0.45,	'CLRKCWF2': +0.45,	'RUGBYR1': +0.448,	'Stanwell': +0.443,	'STAN-1': +0.443,	'STAN-2': +0.443,	'STAN-3': +0.443,	'STAN-4': +0.443,	'BARCALDN': +0.434,	'LILYSF1': +0.434,	'CLERMSF1': +0.434,	'MIDDLSF1': +0.433,	'EMERASF1': +0.426,	'MOUSF1': +0.29,	'GSTONE1': -0.27,	'GSTONE2': -0.27,	'GSTONE5': -0.27,	'GSTONE6': -0.27,	'CALL_B_1': +0.257,	'CALL_B_2': +0.257,	'CPP_3': +0.257,	'CPP_4': +0.257,	'GSTONE3': -0.243,	'GSTONE4': -0.243,	'BUSF1': -0.172,	'CHILDSF1': -0.153}
    prob += pulp.lpSum(Q_NIL_LCCP_BCCP.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index)   <= 1250,"Q>NIL_LCCP_BCCP"

    Q_BCLC_BCCP_CLWU = {'BBATTERY1': +1,	'KIDSPHG1': +0.972,	'KIDSPHG2': +0.972,	'KIDSPHL1': -0.972,	'KIDSPHL2': -0.972,	'BARRON-1': +0.972,	'BARRON-2': +0.972,	'DAYDSF1': +0.972,	'HAYMSF1': +0.972,	'YABULU2': +0.972,	'HAUGHT11': +0.972,	'KAREEYA1': +0.972,	'KAREEYA2': +0.972,	'KAREEYA3': +0.972,	'KAREEYA4': +0.972,	'MSTUART1': +0.972,	'MSTUART2': +0.972,	'MSTUART3': +0.972,	'KSP1': +0.972,	'RRSF1': +0.972,	'KEPWF1': +0.972,	'KEPSF1': +0.972,	'KABANWF1': +0.972,	'YABULU': +0.972,	'SMCSF1': +0.972,	'MEWF1': +0.972,	'CLARESF1': +0.971,	'CSPVPS1': +0.971,	'HAMISF1': +0.971,	'WHITSF1': +0.971,	'BRDDSF01': +0.97,	'BRDDBES1': +0.97,	'CLRKCWF1': +0.97,	'CLRKCWF2': +0.97,	'RUGBYR1': +0.968,	'STABESS1': +0.964,	'STAN-1': +0.964,	'STAN-2': +0.964,	'STAN-3': +0.964,	'STAN-4': +0.964,	'LILYSF1': +0.957,	'BARCALDN': +0.956,	'MIDDLSF1': +0.956,	'CLERMSF1': +0.956,	'EMERASF1': +0.95,	'MOUSF1': +0.833,	'CALL_B_1': +0.804,	'CALL_B_2': +0.804,	'CPP_3': +0.804,	'CPP_4': +0.804,	'GSTONE1': -0.789,	'GSTONE2': -0.789,	'GSTONE5': -0.789,	'GSTONE6': -0.789,	'ALDGASF1': -0.789,	'GSTONE3': -0.779,	'GSTONE4': -0.779,	'BUSF1': -0.525,	'CHILDSF1': -0.478,	'SRSF1': -0.457,	'WOOLES1': -0.319,	'MUCRKSF1': -0.319,	'WOOLGSF1': -0.319}
    prob += pulp.lpSum(Q_BCLC_BCCP_CLWU.get(s.loc[b, "DUID"], 0) * v[b] for b in s.index)   <= 1500,"Q>BCLC_BCCP_CLWU"

    '''
    

'   \n  #SET THE CONSTRAINTS \n  for region in regions:\n     \n      lhs = (pulp.lpSum(v[c] for c in s.index if s.loc[c, "REGIONID"] == region)              + pulp.lpSum(l[d] for d in load.index if load.loc[d, "REGIONID"] == region)) \n  \n      if region == "NSW1":\n          rhs = (flow[\'N_Q_MNSP1\'] + flow[\'NSW1_QLD1\'] - flow[\'VIC1_NSW1\']) + 0.4394*losses[\'NSW1_QLD1\']+0.5028*losses[\'N_Q_MNSP1\']+(1-0.6262)*losses[\'VIC1_NSW1\'] \n      elif region == "QLD1":\n          rhs = (- flow[\'N_Q_MNSP1\'] - flow[\'NSW1_QLD1\']) +(1-0.4394)*losses[\'NSW1_QLD1\']+(1-0.5028)*losses[\'N_Q_MNSP1\']\n      elif region == "SA1":\n          rhs = (- flow[\'V_S_MNSP1\'] - flow[\'V_SA\']) + (1-0.4925)*losses[\'V_SA\']+(1-0.5057)*losses[\'V_S_MNSP1\']\n      elif region == "TAS1":\n          rhs = (flow[\'T_V_MNSP1\'])  +  losses[\'T_V_MNSP1\']\n      else:\n          rhs = (flow[\'VIC1_NSW1\'] + flow[\'V_SA\'] + flow[\'V_S_MNSP1\'] - flow[\'T_V_MNSP1\']) + 0.4925*losses[\'V_SA\']+0.6262*loss